<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Collection

Builds the raw dataset for the study window set in `src/config.py` (`START_DATE`–`END_DATE`, by market expiry):

| Output | Content |
|---|---|
| `data/raw/polymarket_markets.csv` | one row per hourly *"Bitcoin above K on …, H ET?"* market: strike, start/end, YES token, outcome |
| `data/raw/polymarket_prices/YYYY-MM-DD.parquet` | YES-price history at 1-minute fidelity, one file per expiry day |
| `data/raw/btc_1m/YYYY-MM-DD.parquet` | Binance BTC/USDT 1-minute klines (public archive, full history) |
| `data/processed/panel_1m.parquet` | the analysis panel (derived, git-ignored, rebuilt from raw) |

Every step is **resumable**: files that exist are skipped, so after a Colab disconnect just re-run the cells.
For long windows, run the push cell at the end now and then so the downloaded files are not lost.

In [ ]:
# Setup — identical in every notebook; works in Colab and locally
import os, sys, subprocess
BRANCH = "main"   # GitHub branch Colab runs; it must contain src/
if "google.colab" in sys.modules:
    REPO = "/content/DSA-Project"
    URL = "https://github.com/llelus/DSA-Project.git"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL, REPO], check=True)
    else:  # runtime reused: make sure the requested branch is checked out and current
        subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "pull", "-q", "--autostash", "origin", BRANCH], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.isdir("src"):
    raise RuntimeError(f"src/ not found in {os.getcwd()}: branch '{BRANCH}' on GitHub does not "
                       "contain the v2 code yet (push/merge it, or set BRANCH).")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd
from src import config, collect, dataset
from src.results import save_metrics

session = collect.make_session()
print("Study window:", config.START_DATE, "->", config.END_DATE)

## 1. Probe one day

Before the full download, check that the API returns the markets we expect, that `MARKET_PATTERN`
matches their wording, and read the **resolution rules** (which exchange/candle settles the market).
If nothing matches, adjust `MARKET_PATTERN` or the window in `src/config.py`.

Thousands of unrelated markets share the same exact end time, more than the API lets you page
through, so queries are restricted to one **tag**. Cell (a) prints the tags of a known v1 market
(*"Bitcoin above 71,600 on April 9, 4PM ET?"*): if `bitcoin` is not among them, set
`GAMMA_TAG_SLUG` (or `GAMMA_TAG_ID`) in `src/config.py` to one that is.

In [ ]:
# (a) API behaviour and the tag used to filter (fast: a few requests)
day0 = pd.Timestamp(config.START_DATE, tz="UTC")
collect.diagnose_gamma(session, day0)
print()
print(collect.market_tags(session).to_string(index=False))
tag_id = collect.resolve_tag_id(session)
print(f"\nFiltering on tag '{config.GAMMA_TAG_SLUG}' -> tag_id = {tag_id}")

In [ ]:
# (b) Every closed tagged market ending on day0, and how Bitcoin markets are worded
raw = collect.fetch_day_raw(session, day0, day0 + pd.Timedelta(days=1), tag_id=tag_id)
matched = [m for m in raw if collect.parse_market(m)]
templates = collect.question_templates(raw)
print(f"{day0.date()}: {len(raw):,} closed markets with tag {tag_id}, "
      f"{templates.sum():,} mention Bitcoin/BTC, {len(matched):,} match MARKET_PATTERN")
print("\nMost common Bitcoin wordings (numbers -> <N>, months -> <MON>):")
for t, c in templates.head(15).items():
    print(f"  {c:5d} × {t}")
if matched:
    ex = matched[0]
    print(f"\nExample match: {ex['question']}  | start {ex.get('startDate')} | end {ex.get('endDate')}")
    print("Resolution rules:\n", ex.get("description", "")[:1200])

## 2. Market list

If `data/raw/polymarket_markets.csv` already exists it is loaded instead of re-querying the API
(set `REFRESH_MARKETS = True` after changing the study window).

In [ ]:
REFRESH_MARKETS = False
if config.RAW_MARKETS.exists() and not REFRESH_MARKETS:
    markets = collect.load_markets_csv()
    print(f"Loaded {len(markets):,} markets from {config.RAW_MARKETS}")
else:
    markets = collect.fetch_markets(config.START_DATE, config.END_DATE, session=session)
    if markets.empty:
        raise SystemExit("No markets found: adjust START_DATE/END_DATE or MARKET_PATTERN in src/config.py")
    markets = collect.save_markets(markets)
life_min = (markets.market_end - markets.market_start).dt.total_seconds() / 60
print(f"{len(markets):,} markets saved -> {config.RAW_MARKETS}")
print(f"Lifetime (minutes): median {life_min.median():.0f}, 90th pct {life_min.quantile(.9):.0f}")
print("Outcome balance:", markets.result_yes_won.value_counts(dropna=False).to_dict())
markets.head()

## 3. BTC spot — Binance BTC/USDT 1-minute klines

In [ ]:
missing_btc_days = collect.collect_btc(config.START_DATE, config.END_DATE, session=session)

## 4. Polymarket price histories (longest step)

In [ ]:
price_summary = collect.collect_prices(markets, session=session, workers=4)

## 5. Data-quality checks

1. **Settlement rule** — recompute every outcome as *settlement candle close > strike*. Accuracy close to 100%
   confirms that our BTC series is the resolution source and which candle settles the market.
   If the best offset differs from `SETTLE_CANDLE_OFFSET_MIN` in `src/config.py`, change it there.
2. **Price resolution** — spacing between Polymarket price points (v1 data was 10-minute, which made
   minute-level lag claims impossible).
3. **Coverage** — markets without any price history.

In [ ]:
btc = collect.load_btc()
prices = collect.load_prices()

check = dataset.resolution_check(markets, btc, offsets=(-1, 0, 1))
print(check.to_string(index=False))
best = check.loc[check.accuracy.idxmax()]
print(f"Best settlement offset: {int(best.offset_min):+d} min ({best.accuracy:.2%}); "
      f"config uses {config.SETTLE_CANDLE_OFFSET_MIN:+d}")

spacing = prices.groupby("condition_id")["ts"].diff().dt.total_seconds() / 60
print("\nPolymarket point spacing (min):", spacing.describe(percentiles=[.1, .5, .9]).round(2).to_dict())
covered = markets.condition_id.isin(prices.condition_id.unique())
print(f"Markets with price history: {covered.sum():,}/{len(markets):,} ({covered.mean():.1%})")

## 6. Build the analysis panel

In [ ]:
panel = dataset.load_panel(rebuild=True)
print(f"Panel: {len(panel):,} market-minutes, {panel.condition_id.nunique():,} markets")
print(f"Share of minutes with a fresh Polymarket price (<1 min old): {(panel.stale_min < 1).mean():.1%}")
print(f"Rows with fair value defined: {panel.fair.notna().mean():.1%}")

save_metrics("data",
    window=[config.START_DATE, config.END_DATE],
    n_markets=len(markets), n_markets_with_prices=int(covered.sum()),
    n_price_points=len(prices), median_point_spacing_min=float(spacing.median()),
    settle_offset_best=int(best.offset_min), settle_accuracy_best=float(best.accuracy),
    settle_accuracy_config=float(check.set_index("offset_min").loc[config.SETTLE_CANDLE_OFFSET_MIN, "accuracy"]),
    n_btc_days_missing=len(missing_btc_days),
    panel_rows=len(panel), panel_markets=int(panel.condition_id.nunique()),
    share_fresh_price=float((panel.stale_min < 1).mean()))

## 7. Save the raw data to GitHub

Colab's disk is wiped when the runtime is deleted, and the download takes about an hour.
Set `PUSH = True` and run this cell once collection has finished. It needs the Colab secret
`GITHUB_TOKEN`; `GIT_NAME` and `GIT_EMAIL` are optional.

In [ ]:
# Optional: commit outputs back to GitHub (needs Colab secrets GITHUB_TOKEN, GIT_NAME, GIT_EMAIL)
PUSH = False
if PUSH:
    from src.colab import push
    push("data: collect raw Polymarket + BTC data", paths=('data/raw', 'results'))